In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 【读取步骤】
# 读取已经清洗好的 train 和 test
# ============================================================

application_train_cleaned = pd.read_parquet(
    "data/interim/application_train_cleaned.parquet"
)

application_test_cleaned = pd.read_parquet(
    "data/interim/application_test_cleaned.parquet"
)

# 读取之前已经完成的训练集特征表，用于最后核对
application_features_existing = pd.read_parquet(
    "data/processed/application_features.parquet"
)

print(
    "清洗后训练集 Shape:",
    application_train_cleaned.shape
)

print(
    "清洗后测试集 Shape:",
    application_test_cleaned.shape
)


# ============================================================
# 【特征工程函数】
# 建立 application 数值衍生特征
# ============================================================

def create_application_numeric_features(df):

    # 1. 建立副本
    df = df.copy()

    # 2. 贷款收入比
    df["LOAN_TO_INCOME"] = (
        df["AMT_CREDIT"]
        / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    # 3. 年金收入比
    df["ANNUITY_TO_INCOME"] = (
        df["AMT_ANNUITY"]
        / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    # 4. 贷款金额与商品价格比例
    df["CREDIT_TO_GOODS"] = (
        df["AMT_CREDIT"]
        / df["AMT_GOODS_PRICE"].replace(0, np.nan)
    )

    # 5. 工作年限与年龄比例
    df["EMPLOYMENT_TO_AGE"] = (
        df["EMPLOYMENT_YEARS"]
        / df["AGE_YEARS"].replace(0, np.nan)
    )

    # 6. 家庭人均收入
    df["INCOME_PER_PERSON"] = (
        df["AMT_INCOME_TOTAL"]
        / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    )

    # 7. 家庭人均贷款金额
    df["CREDIT_PER_PERSON"] = (
        df["AMT_CREDIT"]
        / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    )

    # 8. 儿童占家庭成员比例
    df["CHILDREN_RATIO"] = (
        df["CNT_CHILDREN"]
        / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    )

    # 9. 家庭人均年金金额
    df["ANNUITY_PER_PERSON"] = (
        df["AMT_ANNUITY"]
        / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    )

    # 10. 支付年金后的剩余收入
    df["REMAINING_INCOME"] = (
        df["AMT_INCOME_TOTAL"]
        - df["AMT_ANNUITY"]
    )

    return df


# ============================================================
# 【分组函数】
# 使用训练集分位点，同时处理 train 和 test
# ============================================================

def add_train_based_quantile_group(
    train_df,
    test_df,
    source_column,
    output_column,
    labels
):

    # 1. 只使用训练集计算分组边界
    _, bins = pd.qcut(
        train_df[source_column],
        q=5,
        retbins=True,
        duplicates="drop"
    )

    # 2. 扩展首尾边界，避免测试集极端值无法分组
    bins[0] = -np.inf
    bins[-1] = np.inf

    # 3. 根据实际区间数量调整标签数量
    used_labels = labels[:len(bins) - 1]

    # 4. 将同一套边界应用到训练集
    train_df[output_column] = pd.cut(
        train_df[source_column],
        bins=bins,
        labels=used_labels,
        include_lowest=True
    )

    # 5. 将同一套边界应用到测试集
    test_df[output_column] = pd.cut(
        test_df[source_column],
        bins=bins,
        labels=used_labels,
        include_lowest=True
    )

    return train_df, test_df


# ============================================================
# 【建立特征】
# 分别建立训练集和测试集数值特征
# ============================================================

application_train_features = create_application_numeric_features(
    application_train_cleaned
)

application_test_features = create_application_numeric_features(
    application_test_cleaned
)


# ============================================================
# 【建立分组特征】
# 所有分组边界只从训练集获得
# ============================================================

group_configs = [
    (
        "LOAN_TO_INCOME",
        "LOAN_TO_INCOME_GROUP",
        [
            "Lowest 20%",
            "20%-40%",
            "40%-60%",
            "60%-80%",
            "Highest 20%"
        ]
    ),
    (
        "ANNUITY_TO_INCOME",
        "ANNUITY_TO_INCOME_GROUP",
        [
            "Lowest 20%",
            "20%-40%",
            "40%-60%",
            "60%-80%",
            "Highest 20%"
        ]
    ),
    (
        "CREDIT_TO_GOODS",
        "CREDIT_TO_GOODS_GROUP",
        [
            "Lowest 20%",
            "20%-40%",
            "40%-60%",
            "60%-80%",
            "Highest 20%"
        ]
    ),
    (
        "EMPLOYMENT_TO_AGE",
        "EMPLOYMENT_TO_AGE_GROUP",
        [
            "Lowest 20%",
            "20%-40%",
            "40%-60%",
            "60%-80%",
            "Highest 20%"
        ]
    ),
    (
        "INCOME_PER_PERSON",
        "INCOME_PER_PERSON_GROUP",
        [
            "Q1 - Lowest20%",
            "Q2:排名20%至40%",
            "Q3:排名40%至60%",
            "Q4:排名60%至80%",
            "Q5 - Highest20%"
        ]
    ),
    (
        "CREDIT_PER_PERSON",
        "CREDIT_PER_PERSON_GROUP",
        [
            "Q1 - Lowest",
            "Q2",
            "Q3",
            "Q4",
            "Q5 - Highest"
        ]
    ),
    (
        "ANNUITY_PER_PERSON",
        "ANNUITY_PER_PERSON_GROUP",
        [
            "Q1 - Lowest",
            "Q2",
            "Q3",
            "Q4",
            "Q5 - Highest"
        ]
    ),
    (
        "REMAINING_INCOME",
        "REMAINING_INCOME_GROUP",
        [
            "Q1 - Lowest",
            "Q2",
            "Q3",
            "Q4",
            "Q5 - Highest"
        ]
    )
]


for source_column, output_column, labels in group_configs:

    (
        application_train_features,
        application_test_features
    ) = add_train_based_quantile_group(
        application_train_features,
        application_test_features,
        source_column,
        output_column,
        labels
    )


# ============================================================
# 【字段对齐】
# 测试集按照训练集字段顺序排列
# ============================================================

train_feature_columns = [
    col
    for col in application_train_features.columns
    if col != "TARGET"
]

application_test_features = application_test_features[
    train_feature_columns
]


# ============================================================
# 【检查步骤】
# 检查行数、主键、字段和 TARGET
# ============================================================

print(
    "新训练集特征 Shape:",
    application_train_features.shape
)

print(
    "新测试集特征 Shape:",
    application_test_features.shape
)

print(
    "训练集重复客户数量:",
    application_train_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

print(
    "测试集重复客户数量:",
    application_test_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

print(
    "训练集包含 TARGET:",
    "TARGET" in application_train_features.columns
)

print(
    "测试集包含 TARGET:",
    "TARGET" in application_test_features.columns
)

print(
    "Train/Test 字段是否一致:",
    train_feature_columns
    == application_test_features.columns.tolist()
)


# ============================================================
# 【核对步骤】
# 与之前生成的训练集特征表比较字段
# ============================================================

missing_columns = sorted(
    set(application_features_existing.columns)
    - set(application_train_features.columns)
)

extra_columns = sorted(
    set(application_train_features.columns)
    - set(application_features_existing.columns)
)

print(
    "相比原特征表缺少的字段:",
    missing_columns
)

print(
    "相比原特征表多出的字段:",
    extra_columns
)


# ============================================================
# 【保存结果】
# 不覆盖原来的 application_features.parquet
# ============================================================

train_output_path = Path(
    "data/processed/application_train_features.parquet"
)

test_output_path = Path(
    "data/processed/application_test_features.parquet"
)

application_train_features.to_parquet(
    train_output_path,
    index=False
)

application_test_features.to_parquet(
    test_output_path,
    index=False
)

print(
    "训练集文件是否存在:",
    train_output_path.exists()
)

print(
    "测试集文件是否存在:",
    test_output_path.exists()
)